In [1]:
%pip install -q --upgrade pageindex

/Users/janibasha/MyProjects/Generative-AI/PageIndex-VectorlessRAG/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pageindex.client import PageIndexClient
from pageindex import PageIndexClient
import pageindex.utils as utils
import os

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.getenv('PAGEINDEX_API_KEY', None)
pi_client: PageIndexClient = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [9]:
from unittest import result
from dotenv import load_dotenv
import os
from langchain_ibm import ChatWatsonx
load_dotenv()

WATSONX_APIKEY = os.getenv('WATSONX_APIKEY', None)
WATSONX_PROJECT_ID = os.getenv('WATSONX_PROJECT_ID', None)
URL = os.getenv("WATSONX_URL","https://us-south.ml.cloud.ibm.com")

def call_llm(input):
    llm = ChatWatsonx(
        model_id="meta-llama/llama-3-3-70b-instruct",
        url = URL,
        apikey = WATSONX_APIKEY,
        project_id = WATSONX_PROJECT_ID,
    )
    result= llm.invoke(input)
    return result


## Step-1. PageIndex Tree Generation


#### Submit the pdf for tree generation

In [4]:
import os, requests

# You can also use our GitHub repo to generate PageIndex tree
# https://github.com/VectifyAI/PageIndex

pdf_url = "https://arxiv.org/pdf/2501.12948.pdf"
pdf_path = os.path.join("../data", pdf_url.split('/')[-1])
os.makedirs(os.path.dirname(pdf_path), exist_ok=True)

response = requests.get(pdf_url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print(f"Downloaded {pdf_url}")

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Downloaded https://arxiv.org/pdf/2501.12948.pdf
Document Submitted: pi-cmnelvbri0axe01pfz38ox0v3


In [6]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of the Document:')
    utils.print_tree(tree)
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'Abstract', 'node_id': '0000', 'summary': 'This text discusses the challenge of gen...'},
 {'title': '1 Introduction',
  'node_id': '0001',
  'summary': 'This text discusses the development of r...'},
 {'title': '2 DeepSeek-R1-Zero',
  'node_id': '0002',
  'summary': 'The text details the training of DeepSee...'},
 {'title': '3. DeepSeek-R1',
  'node_id': '0003',
  'summary': 'This text introduces DeepSeek-R1, an enh...'},
 {'title': '4 Experiment',
  'node_id': '0004',
  'summary': 'The text details the experimental evalua...'},
 {'title': '5 Ethics and Safety Statement',
  'node_id': '0005',
  'summary': 'The text addresses the ethical risks of ...'},
 {'title': '6 Conclusion, Limitation, and Future Wor...',
  'node_id': '0006',
  'summary': 'The text introduces DeepSeek-R1-Zero and...'},
 {'title': '7 Author List',
  'node_id': '0007',
  'summary': 'The text presents an author list organiz...'},
 {'title': 'Appendix A Background'

### Step 2: Reasoning-Based Retrieval with Tree Search

**2.1 Use LLM for tree search and identify nodes that might contain relevant context**

In [11]:
import json

query = "What are the conclusions in this document?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result =  call_llm(search_prompt)

In [ ]:
# remove '```json\n' from both sides of the string
# tree_search_result = tree_search_result.content.replace('```json\n', '')
# tree_search_result = tree_search_result.replace('\n', '')
tree_search_result = tree_search_result.replace('}\n```', '}')
json.loads(tree_search_result)

{'thinking': 'To find the conclusions in the document, we need to look for sections that summarize the main points or findings. The question "What are the conclusions in this document?" suggests we are looking for a summary or overview of the document\'s key takeaways. In academic or technical documents, conclusions are often found in sections titled \'Conclusion\', \'Summary\', \'Findings\', or similar. Given the tree structure, we should focus on nodes with titles that indicate a concluding or summarizing section. Node \'6 Conclusion, Limitation, and Future Work\' directly matches this criteria as it explicitly mentions \'Conclusion\' in its title, suggesting it contains the document\'s conclusions. Other sections like \'Abstract\' might also provide an overview, but they are more about introducing the topic than concluding it. Therefore, the most relevant node for conclusions would be the one titled \'6 Conclusion, Limitation, and Future Work\'.',
 'node_list': ['0006']}

**2.2 Print retrieved nodes and reasoning process**

In [30]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
To find the conclusions in the document, we need to look for sections that summarize the main points
or findings. The question "What are the conclusions in this document?" suggests we are looking for a
summary or overview of the document's key takeaways. In academic or technical documents, conclusions
are often found in sections titled 'Conclusion', 'Summary', 'Findings', or similar. Given the tree
structure, we should focus on nodes with titles that indicate a concluding or summarizing section.
Node '6 Conclusion, Limitation, and Future Work' directly matches this criteria as it explicitly
mentions 'Conclusion' in its title, suggesting it contains the document's conclusions. Other
sections like 'Abstract' might also provide an overview, but they are more about introducing the
topic than concluding it. Therefore, the most relevant node for conclusions would be the one titled
'6 Conclusion, Limitation, and Future Work'.

Retrieved Nodes:
Node ID: 0006	 Page: 10	 Title

### Step 3: Answer Generation


**3.1 Extract relevant context from retrieved nodes**

In [31]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

# 6 Conclusion, Limitation, and Future Work

We present DeepSeek-R1-Zero and DeepSeek-R1, which rely on large-scale RL to incentivize model
reasoning behaviors. Our results demonstrate that pre-trained checkpoints inherently possess
substantial potential for complex reasoning tasks. We believe that the key to unlocking this
potential lies not in large-scale human annotation but in the provision of hard reasoning questions,
a reliable verifier, and sufficient computational resources for reinforcement learning.
Sophisticated reasoning behaviors, such as self-verification and reflection, appeared to emerge
organically during the reinforcement learning process.

Even if DeepSeek-R1 achieves frontier results on reasoning benchmarks, it still faces several
capability limitations, as outlined below:

Structure Output and Tool Use: Currently, the structural output capabilities of DeepSeek-R1 remain
suboptimal compared to existing models. Moreover, DeepSeek-R1 cannot leverag

**3.2 Generate answer based on retrieved context**

In [37]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = call_llm(answer_prompt)
utils.print_wrapped(answer.content)


Generated Answer:

The conclusions in this document are:

1. Pre-trained checkpoints have substantial potential for complex reasoning tasks, which can be
unlocked through large-scale RL, hard reasoning questions, a reliable verifier, and sufficient
computational resources.
2. DeepSeek-R1 achieves frontier results on reasoning benchmarks but still faces limitations in
structure output, token efficiency, language mixing, prompting engineering, and software engineering
tasks.
3. Pure RL methodology presents inherent challenges, such as reward hacking, which can be addressed
by developing innovative approaches to define and refine reward structures.
4. The future holds immense potential for solving complex tasks using pure RL methods, but
challenges remain for tasks where constructing a reliable reward model is difficult.
5. Leveraging tools during the reasoning process holds significant promise for enhancing the scope
and accuracy of machine-driven solutions.
